# Día 2 — El mapa logístico y la ruta al caos

### Taller: Física no lineal en el aula
Congreso de Profesores de Física, Educación Secundaria

---

Ayer vimos que un sistema con ecuaciones sencillas puede volverse impredecible.
Hoy vamos al ejemplo más barato que existe: una sola variable, una sola ecuación,
sin derivadas. Lo que acaban de hacer con la calculadora es exactamente esto:

$$x_{n+1} = r\,x_n\,(1 - x_n)$$

El cuaderno funciona igual que el de ayer: *Copiar en Drive*, después *Entorno de
ejecución → Ejecutar todas*, y la primera celda se toma unos segundos preparando
las figuras. Las figuras vuelven a ser interactivas: deslizadores que recalculan
mientras se arrastran, y un par de gráficos donde se puede hacer click adentro. Si
alguna aparece vacía o sin controles, se vuelve a ejecutar esa celda con
Shift+Enter y listo.

---
## 0. Preparación

Ejecutar esta celda una sola vez. Tarda unos segundos: si la máquina de Colab que
nos tocó no trae el paquete de figuras interactivas, lo instala sola.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import (FloatSlider, IntSlider, Button, HBox, VBox, Label)
from IPython.display import display

try:
    from numba import njit          # compila las cuentas: los deslizadores
except ImportError:                 # responden sin demora
    def njit(f=None, **kw):
        return f if f is not None else (lambda g: g)


def activar_figuras_vivas():
    """Enciende ipympl, que es lo que hace que las figuras respondan al mouse.
    Algunas máquinas de Colab lo traen y otras no, así que si falta lo
    instalamos. Son unos segundos y una sola vez."""
    import importlib, importlib.util, subprocess, sys

    # Primero esto: Colab arma su lista de backends al arrancar, antes de que
    # ipympl exista, y después lo rechaza por no figurar en la lista. Como
    # ipympl se pone de backend apenas se lo importa, el permiso tiene que
    # estar dado antes del import.
    try:
        matplotlib.rcParams.validate["backend"] = lambda s: s
    except Exception:
        pass

    if importlib.util.find_spec("ipympl") is None:
        print("instalando ipympl (sólo la primera vez)...")
        for extra in (["--no-deps"], []):      # sin tocar el resto del entorno
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"]
                           + extra + ["ipympl"], capture_output=True, text=True)
            importlib.invalidate_caches()
            if importlib.util.find_spec("ipympl") is not None:
                break
    try:
        import ipympl
    except Exception as e:
        print("no se pudo cargar ipympl:", type(e).__name__, e)
        return False

    try:
        from google.colab import output      # habilita los widgets externos
        output.enable_custom_widget_manager()
    except ImportError:
        pass

    for nombre in ("widget", "ipympl"):
        try:
            get_ipython().run_line_magic("matplotlib", nombre)
            if "ipympl" in matplotlib.get_backend():
                return True
        except Exception:
            pass
    try:
        matplotlib.use("module://ipympl.backend_nbagg")
        return "ipympl" in matplotlib.get_backend()
    except Exception as e:
        print(type(e).__name__, ":", e)
        return False

VIVO = activar_figuras_vivas()

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.ioff()                          # cada figura se muestra donde la pongamos


def lienzo(ancho=7.0, alto=4.2, barra=False):
    fig, ax = plt.subplots(figsize=(ancho, alto))
    fig.canvas.header_visible = False
    fig.canvas.toolbar_visible = barra
    fig.canvas.footer_visible = barra
    return fig, ax

def mostrar(fig, *controles):
    if controles:
        display(VBox(list(controles)))
    display(fig.canvas if VIVO else fig)


# --- los dos mapas del día. El seno aparece recién en el Ejercicio 1 ---
@njit
def iterar(x, r, cual=0):
    if cual == 0:
        return r*x*(1.0 - x)            # el mapa logístico
    return r*np.sin(np.pi*x)            # el mapa seno, para el ejercicio 1

def logistica(x, r):
    return r*x*(1 - x)

iterar(0.3, 3.0, 0)                     # primera llamada: compila

if VIVO:
    # Colab baja de internet el motor que dibuja estas figuras, y recién lo
    # hace cuando aparece la primera. Le damos una de prueba y unos segundos
    # de ventaja: si no, en un "Ejecutar todas" las primeras salen mudas.
    import time
    prueba, ejes = plt.subplots(figsize=(4.8, 0.8))
    ejes.text(0.5, 0.5, "si ves este cartel, las figuras interactivas andan",
              ha="center", va="center", fontsize=9)
    ejes.axis("off")
    prueba.canvas.header_visible = False
    prueba.canvas.toolbar_visible = False
    prueba.canvas.footer_visible = False
    display(prueba.canvas)
    time.sleep(8)
    print("Todo listo. Figuras interactivas activadas:", matplotlib.get_backend())
else:
    print("ATENCIÓN: las figuras interactivas no arrancaron.")
    print("Probá 'Entorno de ejecución → Reiniciar y ejecutar todo'.")
    print("Si sigue igual, el cuaderno funciona de todos modos, pero las figuras")
    print("quedan fijas en los valores que trae cada deslizador.")

---
## 1. De dónde sale esta cuenta

La ecuación de hoy no salió de la nada. Es el modelo más simple para una población
que se reproduce **por generaciones separadas**: los insectos que ponen huevos y
mueren, los salmones que desovan una vez al año, las plantas anuales. No hay
solapamiento entre padres e hijos, así que tiene sentido preguntarse cuántos habrá
el año que viene en función de cuántos hay este año, y no cómo cambia la población
instante a instante.

Si este año hay $N_n$ individuos y cada uno deja $r$ descendientes, tendríamos
$N_{n+1} = r\,N_n$, que es crecimiento exponencial y se va al infinito. En el mundo
real hay un techo: comida, espacio, luz. Verhulst, en 1838, propuso frenar el
crecimiento con un factor que se apague cuando la población se acerca a ese techo.

Conviene medir la población **como fracción del máximo posible**, y llamarla $x$,
que entonces va de 0 a 1. El freno más simple que se apaga en $x = 1$ es
justamente $(1 - x)$:

$$x_{n+1} = r\,x_n\,(1 - x_n)$$

Con $x$ chico el freno casi no actúa y la población se multiplica por $r$; cuando
$x$ se acerca a 1, el freno la aplasta. Nada más que eso.

Dos cosas para entender los límites de $r$. Como la parábola tiene su máximo en
$x = 0.5$ y ahí vale $r/4$, si tomamos $r > 4$ la población se pasa de 1, el freno
se vuelve negativo y todo se va a menos infinito. Así que $r$ va de 0 a 4, y ese es
todo el universo que vamos a explorar hoy.

In [ ]:
#@title La familia de parábolas { display-mode: "form" }
fig, ax = lienzo(6.4, 4.6)
x = np.linspace(0, 1, 400)
for r in [0.8, 1.6, 2.4, 3.2, 4.0]:
    ax.plot(x, logistica(x, r), lw=2, label=f"r = {r}")
ax.plot(x, x, lw=1.3, ls="--", color="gray", label="y = x")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("población de este año  (xₙ)")
ax.set_ylabel("población del año próximo  (xₙ₊₁)")
ax.set_title("La misma parábola, más alta o más baja según r")
ax.legend(fontsize=9, loc="upper right")
mostrar(fig)

Una aclaración que vale la pena hacer en clase, porque es una trampa habitual: la
versión **continua** de este modelo, la que se resuelve con una ecuación
diferencial y da la clásica curva en forma de S, no hace nada de lo que vamos a ver
hoy. Crece, se acerca al techo y se queda ahí, y punto. Todo lo que sigue aparece
porque la población avanza **a los saltos**, una generación entera por vez, y ese
retardo la hace pasarse de largo.

Robert May, en 1976, publicó un artículo en *Nature* llamado *Simple mathematical
models with very complicated dynamics*, donde mostró todo esto y terminaba pidiendo
que se enseñara temprano, porque la intuición de que "ecuación simple = comportamiento
simple" es falsa y hace daño. Cincuenta años después estamos haciéndole caso.

---
## 2. Lo que acaban de hacer con la calculadora

Recién hicieron a mano lo mismo que hace la celda de abajo: arrancar con un valor,
multiplicarlo por $r$ y por $(1-x)$, y repetir. Vamos a comparar.

El gráfico muestra $x_n$ contra $n$, que es la misma lista de números que
anotaron, puesta en un eje. Abajo del gráfico el programa dice qué encontró: si la
sucesión se estabiliza, si entra en un ciclo y de qué período, o si no se repite
nunca.

In [ ]:
def orbita(r, x0, n, cual=0):
    "La sucesión completa, paso a paso: lo mismo que hicieron a mano."
    xs = np.empty(n + 1)
    xs[0] = x0
    for i in range(n):
        xs[i+1] = iterar(xs[i], r, cual)
    return xs


@njit
def periodo(r, cual=0, n_trans=20000, tol=1e-8):
    """Cada cuántos pasos se repite la órbita a largo plazo. Primero itera un
    buen rato para que el sistema se olvide del arranque, y después compara
    cada valor con el que viene p pasos después. La vamos a volver a usar."""
    x = 0.3
    for _ in range(n_trans):
        x = iterar(x, r, cual)
    v = np.empty(64)
    for i in range(64):
        x = iterar(x, r, cual)
        v[i] = x
    for p in range(1, 33):
        igual = True
        for i in range(64 - p):
            if abs(v[i+p] - v[i]) > tol:
                igual = False
                break
        if igual:
            return p
    return 0                            # no se repite: caos

In [ ]:
#@title La sucesión xₙ, con deslizadores { display-mode: "form" }
s_r  = FloatSlider(min=1.0, max=4.0, step=0.001, value=2.8, description="r",
                   continuous_update=True, readout_format=".3f")
s_x0 = FloatSlider(min=0.01, max=0.99, step=0.01, value=0.30, description="x₀",
                   continuous_update=True, readout_format=".2f")
s_n  = IntSlider(min=10, max=120, value=40, description="pasos",
                 continuous_update=True)
diagnostico = Label("")

fig, ax = lienzo(8.0, 4.2)
linea, = ax.plot([], [], "o-", ms=4, lw=1, color="#1f4e79")
ax.set_xlabel("n (generación)"); ax.set_ylabel("xₙ")
ax.set_ylim(-0.02, 1.02)

def dibujar(_=None):
    r, x0, n = s_r.value, s_x0.value, s_n.value
    xs = orbita(r, x0, n)
    linea.set_data(np.arange(n + 1), xs)
    ax.set_xlim(-1, n + 1)
    ax.set_title(f"r = {r:.3f}     x₀ = {x0:.2f}")
    p = periodo(r)
    if p == 0:
        diagnostico.value = "no se repite nunca: caos"
    elif p == 1:
        diagnostico.value = f"se estabiliza en un solo valor: x = {xs[-1]:.6f}"
    else:
        ult = sorted(set(np.round(xs[-2*p:], 5).tolist()))
        diagnostico.value = (f"ciclo de período {p}. Últimos valores: "
                             + ", ".join(f"{v:.4f}" for v in ult))
    fig.canvas.draw_idle()

for s in (s_r, s_x0, s_n):
    s.observe(dibujar, names="value")
dibujar()
mostrar(fig, HBox([s_r, s_x0]), HBox([s_n, diagnostico]))

**Para probar:** recorran los valores que les tocaron con la calculadora: 2.8, 3.2,
3.5, 3.9. Después pasen el deslizador de $r$ despacio desde 2.8 hasta 4 y miren
cómo cambia el mensaje de abajo: un valor fijo, después dos, después cuatro, ocho,
y en algún momento deja de repetirse. Eso es una **cascada de duplicación de
período**, y es lo que vamos a mirar de cerca en un rato.

Fíjense en lo más importante: **la ecuación no cambió nunca**. Lo único que se
movió es un número.

Dos observaciones sobre las cuentas a mano:

- Si arrancaron desde valores distintos de $x_0$ y les dio lo mismo al final, no es
  casualidad: muevan el deslizador de $x_0$ y van a ver que el destino a largo plazo
  casi no depende de dónde se arranca. Eso es un atractor, el mismo concepto de ayer.
- Si con $r = 3.9$ compararon los números con el de al lado y después de unos
  quince o veinte pasos ya no coincidían, tampoco se equivocaron. Cada uno redondeó
  a las cifras que mostraba su calculadora, y esa diferencia minúscula se fue
  agrandando sola. Es la sensibilidad a las condiciones iniciales de ayer, ahora en
  una calculadora de bolsillo.

**Para comparar con lo suyo.** Pongan sus números en la lista de abajo y ejecuten
la celda. Los puntos rojos son lo que anotaron a mano; la línea azul es lo que
calcula la computadora, arrancando del mismo $x_0$.

In [ ]:
mis_valores = [0.3, 0.63, 0.70, 0.59]      # ←←← ESCRIBIR ACÁ lo que anotaron
mi_r        = 3.0                          # ←←← y con qué r lo hicieron

# ---- de acá para abajo no hace falta tocar nada ----
fig, ax = lienzo(7.6, 4.0)
n = max(len(mis_valores) + 4, 12)
xs = orbita(mi_r, mis_valores[0], n)
ax.plot(np.arange(n + 1), xs, "-o", ms=4, lw=1.2, color="#1f4e79",
        label="la computadora")
ax.plot(np.arange(len(mis_valores)), mis_valores, "o", ms=9, color="#c0392b",
        label="lo que anotaron")
ax.set_xlabel("n"); ax.set_ylabel("xₙ"); ax.set_ylim(-0.02, 1.02)
ax.set_title(f"r = {mi_r}")
ax.legend(fontsize=9)
mostrar(fig)

dif = [abs(mis_valores[i] - xs[i]) for i in range(len(mis_valores))]
print(f"diferencia máxima con la computadora: {max(dif):.2e}")

---
## 3. La telaraña

<font color="#1a73e8"><b>Sección opcional.</b></font> Es la forma más linda de ver
qué está pasando, pero si andamos con el tiempo justo se puede saltear sin perder
el hilo: lo que sigue después no depende de esta sección.

La idea es hacer la iteración con lápiz en vez de con números. Dibujamos dos curvas,
la parábola $y = r\,x(1-x)$ y la diagonal $y = x$. Para iterar: desde $x_n$ subimos
vertical hasta la parábola, y eso nos da $x_{n+1}$; después nos movemos horizontal
hasta la diagonal, que convierte ese resultado en la nueva entrada. Repetir.

Los **puntos fijos** son los valores que no cambian al iterar, o sea donde la
parábola cruza la diagonal. Se los puede calcular a mano:

$$x^* = r\,x^*(1 - x^*) \;\;\Longrightarrow\;\; x^* = 0
\quad\text{o}\quad x^* = 1 - \frac{1}{r}$$

Que el sistema se quede ahí o se escape depende de la **pendiente de la parábola**
en el cruce. Derivando, $f'(x) = r\,(1 - 2x)$, y evaluando en el punto fijo:

$$f'(x^*) = r\left(1 - 2\left(1 - \tfrac{1}{r}\right)\right) = 2 - r$$

La regla es que el punto fijo atrae si $|f'(x^*)| < 1$ y repele si es mayor que 1.
Acá eso da $|2 - r| < 1$, o sea $1 < r < 3$. Esa cuenta de tres renglones predice
exactamente dónde se termina el comportamiento simple.

In [ ]:
#@title El diagrama de telaraña { display-mode: "form" }
s_r3 = FloatSlider(min=1.0, max=4.0, step=0.001, value=2.8, description="r",
                   continuous_update=True, readout_format=".3f")
s_x3 = FloatSlider(min=0.01, max=0.99, step=0.01, value=0.20, description="x₀",
                   continuous_update=True, readout_format=".2f")
s_p3 = IntSlider(min=1, max=120, value=40, description="pasos",
                 continuous_update=True)

fig, ax = lienzo(5.8, 5.8)
xx = np.linspace(0, 1, 400)
parabola, = ax.plot([], [], lw=2, color="#1f4e79")
ax.plot(xx, xx, lw=1.2, color="gray", ls="--")
telarana, = ax.plot([], [], lw=0.9, color="#c0392b")
punto_fijo, = ax.plot([], [], "ko", ms=8, zorder=5)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.set_xlabel("xₙ"); ax.set_ylabel("xₙ₊₁")

def dibujar(_=None):
    r, x0, pasos = s_r3.value, s_x3.value, s_p3.value
    parabola.set_data(xx, logistica(xx, r))

    px, py = [x0], [0.0]                 # el camino de la telaraña
    xi = x0
    for _ in range(pasos):
        yi = logistica(xi, r)
        px += [xi, yi]
        py += [yi, yi]
        xi = yi
    telarana.set_data(px, py)

    if r > 1:
        xf = 1 - 1/r
        pend = abs(2 - r)
        punto_fijo.set_data([xf], [xf])
        estado = "atrae" if pend < 1 else "repele"
        ax.set_title(f"r = {r:.3f}     x* = {xf:.4f}     "
                     f"|f'(x*)| = {pend:.3f}, {estado}")
    else:
        punto_fijo.set_data([], [])
        ax.set_title(f"r = {r:.3f}     la población se extingue")
    fig.canvas.draw_idle()

for s in (s_r3, s_x3, s_p3):
    s.observe(dibujar, names="value")
dibujar()
mostrar(fig, HBox([s_r3, s_x3]), HBox([s_p3]))

**Para probar:**

1. Con $r = 2.8$ la telaraña se enrosca hacia el cruce y se queda ahí. La pendiente
   vale 0.8, menor que 1.
2. Vayan subiendo $r$ de a poco hasta pasar 3. La pendiente cruza el 1, el punto
   fijo pasa a repeler y la telaraña se estabiliza en un **cuadrado** entre dos
   valores: ése es el período 2.
3. En $r = 3.5$ el cuadrado se convierte en una figura más complicada, de período 4.
4. En $r = 3.9$ la telaraña llena todo y no se cierra nunca.

El momento exacto en que aparece el cuadrado es $r = 3$, que es cuando
$f'(x^*) = 2 - r$ vale $-1$. A ese cambio de comportamiento se lo llama
**bifurcación**, y acabamos de calcularlo con lápiz y papel.

---
## 4. El diagrama de bifurcación

En vez de mirar un $r$ por vez, mirémoslos todos juntos. Para cada valor de $r$:

1. iteramos muchas veces y **tiramos** los primeros pasos, que son el transitorio;
2. dibujamos los valores que quedan, uno arriba del otro en la columna de ese $r$.

Si el sistema termina en un punto fijo, esa columna tiene un punto. Si termina en
un ciclo de período 2, tiene dos. Si es caótico, la columna se llena.

A la izquierda está el pedazo que elijan con los deslizadores, recalculado cada vez.
A la derecha, el diagrama completo con un recuadro rojo que muestra dónde están
parados. La raya punteada marca el centro de la ventana, y arriba dice a qué $r$
corresponde: eso nos va a servir para medir.

In [ ]:
def densidad(r0, r1, x0, x1, cual=0, nr=800, nx=600,
             n_trans=3000, n_guardar=400):
    """Cuenta cuántas veces cae la órbita en cada casillero de la imagen.
    Es mucho más rápido que dibujar medio millón de puntos sueltos."""
    r = np.linspace(r0, r1, nr)
    x = np.full(nr, 0.3)
    for _ in range(n_trans):
        x = iterar(x, r, cual)
    columnas = np.arange(nr)
    cae = []
    for _ in range(n_guardar):
        x = iterar(x, r, cual)
        fila = ((x - x0)/(x1 - x0)*nx).astype(np.int64)
        dentro = (fila >= 0) & (fila < nx)
        cae.append(fila[dentro]*nr + columnas[dentro])
    H = np.bincount(np.concatenate(cae), minlength=nx*nr).reshape(nx, nr)
    return np.sqrt(H.astype(float))

def contraste(H):
    positivos = H[H > 0]
    return np.quantile(positivos, 0.90) if positivos.size else 1.0

In [ ]:
#@title Diagrama de bifurcación con zoom { display-mode: "form" }
COMPLETO = (2.5, 4.0, 0.0, 1.0)

s_rd = FloatSlider(min=2.5, max=4.0, step=0.0001, value=2.5, readout_format=".4f",
                   description="r desde", continuous_update=True)
s_rh = FloatSlider(min=2.5, max=4.0, step=0.0001, value=4.0, readout_format=".4f",
                   description="r hasta", continuous_update=True)
s_xd = FloatSlider(min=0.0, max=1.0, step=0.001, value=0.0, readout_format=".3f",
                   description="x desde", continuous_update=True)
s_xh = FloatSlider(min=0.0, max=1.0, step=0.001, value=1.0, readout_format=".3f",
                   description="x hasta", continuous_update=True)

fig, (az, ac) = plt.subplots(1, 2, figsize=(12.5, 5.0),
                             gridspec_kw={"width_ratios": [1.35, 1]})
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.footer_visible = False

H0 = densidad(*COMPLETO)
im_zoom = az.imshow(H0, origin="lower", extent=COMPLETO, aspect="auto",
                    cmap="gray_r", interpolation="antialiased",
                    vmin=0, vmax=contraste(H0))
centro = az.axvline(3.25, ls="--", lw=1.2, color="#c0392b")
az.set_xlabel("r"); az.set_ylabel("x"); az.grid(False)

ac.imshow(H0, origin="lower", extent=COMPLETO, aspect="auto", cmap="gray_r",
          interpolation="antialiased", vmin=0, vmax=contraste(H0))
recuadro, = ac.plot([], [], lw=1.6, color="#c0392b")
ac.set_xlim(2.5, 4.0); ac.set_ylim(0, 1)
ac.set_xlabel("r"); ac.set_title("dónde estamos parados", fontsize=10)
ac.grid(False)

def ventana():
    r0, r1 = sorted((s_rd.value, s_rh.value))
    x0, x1 = sorted((s_xd.value, s_xh.value))
    r1 = max(r1, r0 + 1e-5)
    x1 = max(x1, x0 + 1e-4)
    return r0, r1, x0, x1

def dibujar(_=None):
    r0, r1, x0, x1 = ventana()
    H = densidad(r0, r1, x0, x1)
    im_zoom.set_data(H)
    im_zoom.set_extent((r0, r1, x0, x1))
    im_zoom.set_clim(0, contraste(H))
    az.set_xlim(r0, r1); az.set_ylim(x0, x1)
    c = 0.5*(r0 + r1)
    centro.set_xdata([c, c])
    az.set_title(f"centro de la ventana:  r = {c:.5f}      "
                 f"ancho = {r1-r0:.1e}")
    recuadro.set_data([r0, r1, r1, r0, r0], [x0, x0, x1, x1, x0])
    fig.canvas.draw_idle()

def ir_a(r0, r1, x0, x1):
    def apretado(_):
        for s in (s_rd, s_rh, s_xd, s_xh):
            s.unobserve(dibujar, names="value")
        s_rd.value, s_rh.value = r0, r1
        s_xd.value, s_xh.value = x0, x1
        for s in (s_rd, s_rh, s_xd, s_xh):
            s.observe(dibujar, names="value")
        dibujar()
    return apretado

atajos = [("todo", COMPLETO),
          ("1ª duplicación", (2.95, 3.05, 0.60, 0.72)),
          ("2ª duplicación", (3.43, 3.47, 0.40, 0.60)),
          ("la cascada", (3.540, 3.572, 0.30, 0.70)),
          ("período 3", (3.82, 3.87, 0.10, 0.60))]
botones = []
for nombre, vista in atajos:
    b = Button(description=nombre, layout={"width": "130px"})
    b.on_click(ir_a(*vista))
    botones.append(b)

for s in (s_rd, s_rh, s_xd, s_xh):
    s.observe(dibujar, names="value")
dibujar()
mostrar(fig, HBox([s_rd, s_rh]), HBox([s_xd, s_xh]), HBox(botones))

**Para probar.** Los botones de abajo llevan directo a los lugares interesantes;
los deslizadores permiten moverse con libertad. Se puede escribir un número exacto
haciendo click sobre el valor de cada deslizador.

1. Empiecen con la vista completa. De izquierda a derecha: una sola línea hasta
   $r = 3$, que se parte en dos; en $r \approx 3.449$ cada rama se parte de nuevo;
   después en ocho, en dieciséis, cada vez más seguido, hasta que alrededor de
   $r \approx 3.57$ se vuelve un borrón. Ahí empieza el caos.
2. Adentro del caos hay **franjas blancas**: valores de $r$ donde el sistema vuelve
   a ser ordenado. La más ancha está cerca de 3.83 y es la ventana de período 3.
3. Ahora la parte más linda. Vayan al botón "la cascada" y sigan achicando la
   ventana alrededor de cualquier rama, comparando con el diagrama completo de la
   derecha. La estructura que aparece es **la misma**, más chica y un poco
   deformada. Entren en la ventana de período 3 y van a encontrar, adentro, otro
   diagrama de bifurcación completo con su propia cascada.

Eso se llama **autosemejanza**: la misma forma reaparece a todas las escalas. Y no
es una curiosidad estética. Es la razón de que exista el número de la sección que
sigue.

---
## 5. La constante de Feigenbaum

Llamemos $r_1 = 3$ al valor donde el punto fijo se parte en dos, $r_2$ donde cada
rama se vuelve a partir, $r_3$ la siguiente, y así. Ya vieron que se van juntando
cada vez más rápido. Feigenbaum, a fines de los 70, se preguntó **a qué ritmo**, y
calculó el cociente entre huecos sucesivos:

$$\delta_n = \frac{r_n - r_{n-1}}{r_{n+1} - r_n}$$

Lo que encontró es que ese cociente tiende a un número fijo,

$$\delta = 4.669201609\ldots$$

y, lo que es mucho más raro, que ese número **no depende del mapa**. Sale igual
para la parábola, para el seno, para el goteo de una canilla y para un circuito
electrónico. Es una constante universal, como $\pi$, pero de las bifurcaciones.

### Primero, a mano

Usen el diagrama de arriba para medir los primeros $r_n$. Para cada uno, acerquen
la ventana hasta que la bifurcación quede sobre la raya punteada del centro, y
anoten el número que aparece en el título.

Los dos primeros son fáciles. El tercero y el cuarto requieren paciencia, porque
están cada vez más cerca.

La celda de abajo viene con valores medidos a las apuradas, con dos decimales.
Ejecútenla así como está para ver qué sale, y después reemplácenlos por los suyos,
con la mayor cantidad de decimales que hayan podido leer. Van a ver que la
precisión importa bastante.

In [ ]:
mis_r = [3.00, 3.45, 3.54, 3.56]      # ←←← ESCRIBIR ACÁ lo que midieron

# ---- de acá para abajo no hace falta tocar nada ----
print("huecos entre bifurcaciones:")
huecos = [mis_r[i+1] - mis_r[i] for i in range(len(mis_r) - 1)]
for i, h in enumerate(huecos, start=1):
    print(f"   r{i+1} − r{i} = {h:.4f}")

print("\ncociente entre huecos sucesivos:")
for i in range(len(huecos) - 1):
    print(f"   δ{i+1} = {huecos[i]/huecos[i+1]:.3f}")
print("\nvalor exacto: δ = 4.669201609...")

### Después, con un algoritmo

Medir a ojo tiene un límite obvio. Vamos a automatizar la misma idea, y el programa
no es más complicado que lo que acaban de hacer a mano. Son dos funciones.

La primera ya la usamos en la sección 2: dado un $r$, itera un buen rato para que
el sistema se olvide del arranque, y después se fija cada cuántos pasos se repite.
Eso es el período.

La segunda busca la bifurcación **partiendo el intervalo al medio**. Si en el
extremo izquierdo el período es 2 y en el derecho es 4, la bifurcación está en el
medio: miramos el punto del centro, nos quedamos con la mitad donde el período
cambia, y repetimos. Cincuenta veces alcanzan de sobra.

In [ ]:
def bifurcacion(p, r_izq, r_der, cual=0):
    "El r donde el período pasa de p a 2p, buscado partiendo el intervalo."
    for _ in range(50):
        medio = 0.5*(r_izq + r_der)
        q = periodo(medio, cual)
        if q != 0 and q <= p:          # todavía estamos antes de la bifurcación
            r_izq = medio
        else:                          # ya estamos después
            r_der = medio
    return 0.5*(r_izq + r_der)


def cascada(r_ini, r_fin, cuantas=6, cual=0):
    """Encuentra r1, r2, r3, ... Los dos primeros se buscan en todo el rango.
    De ahí en más se busca en una ventana chica apoyada en el hueco anterior,
    porque los huecos se achican rápido y más a la derecha hay ventanas
    periódicas que confundirían la búsqueda."""
    r1 = bifurcacion(1, r_ini, r_fin, cual)
    r2 = bifurcacion(2, r1 + 1e-12, r_fin, cual)
    puntos = [r1, r2]
    for k in range(2, cuantas):
        hueco = puntos[-1] - puntos[-2]
        puntos.append(bifurcacion(2**k, puntos[-1] + 1e-12,
                                  min(r_fin, puntos[-1] + 0.4*hueco), cual))
    return puntos

In [ ]:
EXACTO = 4.669201609102990

rs = cascada(2.5, 4.0, 6)
print("  n    período      r_n detectado")
for i, r in enumerate(rs, start=1):
    print(f"  {i}    {2**(i-1):3d} → {2**i:<4d}  {r:.6f}")

print("\n  cociente entre huecos sucesivos:")
for i in range(1, len(rs) - 1):
    d = (rs[i] - rs[i-1])/(rs[i+1] - rs[i])
    print(f"    δ{i} = {d:.4f}     (se aparta {abs(d-EXACTO)/EXACTO*100:.1f} %)")
print(f"\n  valor exacto: δ = {EXACTO}")

Los cocientes dan 4.75, 4.65, 4.65 y 4.64: rondan el valor verdadero con un error
del orden del uno por ciento, pero se quedan ahí dando vueltas en vez de converger
prolijamente. Vale la pena entender por qué, porque el motivo es del sistema y no
un error de programación.

Cerca de una bifurcación, el sistema tarda **muchísimo** en acomodarse. Justo en el
punto de bifurcación la convergencia deja de ser exponencial y se vuelve lentísima,
así que por más que iteremos veinte mil veces, la órbita todavía se está moviendo
un poquito. Nuestro detector de período, que compara con una tolerancia, ve ese
temblor y declara la bifurcación un poco antes de donde está. El error es
sistemático: siempre para el mismo lado.

Se puede achicar iterando más (probá cambiar `n_trans` a 50000 en la función
`periodo` y volver a correr: los $\delta$ mejoran), pero para sacar los seis
decimales de Feigenbaum hace falta otro método, que en vez de buscar las
bifurcaciones busca los ciclos **superestables**, que son mucho más fáciles de
localizar con precisión. Con eso se llega a 4.669191 sin despeinarse.

Para lo que nos importa hoy, esto alcanza y sobra: el número aparece, es del orden
de 4.67, y no se movió de ahí.

---
## Ejercicio 1 — la universalidad

Acá está la parte fuerte del día. Si $\delta$ fuera una propiedad de la parábola,
sería una curiosidad. Lo que dice Feigenbaum es que **no depende del mapa**,
siempre que tenga un solo máximo y sea suave.

Probémoslo con uno que no tiene nada que ver con una parábola:

$$x_{n+1} = r\,\sin(\pi x_n)$$

La celda de abajo corre exactamente el mismo algoritmo sobre los dos mapas y los
pone lado a lado. No hay que cambiar nada, sólo ejecutarla y comparar las dos
columnas de números.

In [ ]:
#@title Los dos mapas, al mismo tiempo { display-mode: "form" }
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.footer_visible = False

for ax, cual, vista, nombre in [
        (a1, 0, (2.5, 4.0, 0.0, 1.0), "logístico:  r·x·(1−x)"),
        (a2, 1, (0.5, 1.0, 0.0, 1.0), "seno:  r·sen(πx)")]:
    H = densidad(*vista, cual=cual)
    ax.imshow(H, origin="lower", extent=vista, aspect="auto", cmap="gray_r",
              interpolation="antialiased", vmin=0, vmax=contraste(H))
    ax.set_xlabel("r"); ax.set_ylabel("x"); ax.set_title(nombre, fontsize=11)
    ax.grid(False)
mostrar(fig)

log = cascada(2.5, 4.0, 6, cual=0)
sen = cascada(0.5, 1.0, 6, cual=1)

print("        r_n del logístico      r_n del seno")
for i, (a, b) in enumerate(zip(log, sen), start=1):
    print(f"  r{i}       {a:.6f}              {b:.6f}")

print("\n        δ del logístico       δ del seno")
for i in range(1, len(log) - 1):
    d1 = (log[i] - log[i-1])/(log[i+1] - log[i])
    d2 = (sen[i] - sen[i-1])/(sen[i+1] - sen[i])
    print(f"  δ{i}       {d1:.4f}                {d2:.4f}")
print(f"\n  exacto: {EXACTO:.6f}")

Miren bien las dos tablas. Los dos diagramas se parecen pero **no son el mismo**:
el seno vive en otro rango de $r$, sus ramas están en otros lugares y sus
bifurcaciones ocurren en números completamente distintos. La primera bifurcación
del logístico es en 3 y la del seno en 0.72; no tienen nada que ver.

Y sin embargo los cocientes convergen al mismo número.

Ése es el resultado profundo del día: la ruta al caos por duplicación de período
tiene una estructura cuantitativa que **no depende de los detalles del sistema**.
Por eso el mismo 4.669 aparece midiendo convección en helio líquido, un circuito
con un diodo o una canilla que gotea, y por eso tiene sentido enseñarlo con una
parábola.

---
## Ejercicio 2 — la ventana de período 3

Volvamos a las franjas blancas adentro del caos. La más ancha está cerca de
$r = 3.83$, y adentro de ella el sistema es perfectamente ordenado: la población
alterna entre tres valores, para siempre.

Abajo hay dos paneles. A la izquierda, el diagrama de bifurcación de esa zona;
pueden hacer **click adentro** para elegir un valor de $r$, o usar el deslizador. A
la derecha, la sucesión $x_n$ para ese $r$, después de tirar el transitorio. El
título dice qué período encontró el programa.

Busquen el rango donde aparece el período 3, y después métanse justo en los bordes.

In [ ]:
#@title La ventana de período 3 { display-mode: "form" }
VENTANA = (3.80, 3.88, 0.0, 1.0)

s_rv = FloatSlider(min=VENTANA[0], max=VENTANA[1], step=0.0002, value=3.835,
                   description="r", continuous_update=True, readout_format=".4f")

fig, (az, ad) = plt.subplots(1, 2, figsize=(12.5, 4.6))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.footer_visible = False

Hv = densidad(*VENTANA, n_trans=4000)
az.imshow(Hv, origin="lower", extent=VENTANA, aspect="auto", cmap="gray_r",
          interpolation="antialiased", vmin=0, vmax=contraste(Hv))
marca = az.axvline(3.835, lw=1.4, color="#c0392b")
az.set_xlim(VENTANA[0], VENTANA[1]); az.set_ylim(0, 1)
az.set_xlabel("r"); az.set_ylabel("x"); az.grid(False)
az.set_title("hacé click para elegir r", fontsize=10)

serie, = ad.plot([], [], "o-", ms=4, lw=1, color="#1f4e79")
ad.set_xlabel("n"); ad.set_ylabel("xₙ"); ad.set_ylim(0, 1); ad.set_xlim(0, 60)

def dibujar(_=None):
    r = s_rv.value
    marca.set_xdata([r, r])
    x = 0.3
    for _ in range(4000):
        x = iterar(x, r)
    xs = [x]
    for _ in range(60):
        x = iterar(x, r)
        xs.append(x)
    serie.set_data(np.arange(len(xs)), xs)
    p = periodo(r)
    ad.set_title(f"r = {r:.4f}   →   "
                 + ("sin período: caos" if p == 0 else f"período {p}"),
                 fontsize=11)
    fig.canvas.draw_idle()

def al_click(ev):
    if ev.inaxes is az and ev.button == 1:
        s_rv.value = min(max(ev.xdata, VENTANA[0]), VENTANA[1])

fig.canvas.mpl_connect("button_press_event", al_click)
s_rv.observe(dibujar, names="value")
dibujar()
mostrar(fig, HBox([s_rv]))

**Para probar:**

1. Barran $r$ desde 3.80 hacia arriba. Hasta 3.8284 el título dice caos y la
   sucesión de la derecha no repite nada. Justo ahí, de golpe, aparece el período 3:
   la ventana arranca exactamente en $1 + \sqrt{8} = 3.8284\ldots$, un número que se
   puede calcular a mano exigiendo que $f^3$ toque la diagonal.
2. Adentro de la ventana, la sucesión de la derecha salta entre tres valores. Miren
   que el sistema sigue siendo el mismo de siempre, con la misma parábola.
3. Sigan subiendo. Alrededor de 3.8415 el período 3 se parte en 6, después en 12: la
   misma cascada de antes, adentro de la ventana, y termina en caos otra vez.
4. Vuelvan a un valor apenas **por debajo** de 3.8284, por ejemplo 3.8280, y miren
   la sucesión de la derecha con atención. Se ve casi periódica durante varios pasos
   y de repente se desordena, y después vuelve a ordenarse. Eso se llama
   **intermitencia**, y es otra de las rutas clásicas al caos.

> El **teorema de Sharkovskii** dice algo notable sobre esto: si un mapa continuo
> tiene una órbita de período 3, entonces tiene órbitas de **todos** los períodos.
> De ahí el título del trabajo de Li y Yorke de 1975, *Period three implies chaos*,
> que además es el artículo que le puso el nombre al campo.

---
## 6. Para llevarse

| Concepto | Dónde apareció |
|---|---|
| Iteración y punto fijo | la calculadora, la telaraña |
| Estabilidad según la pendiente | $\lvert f'(x^*)\rvert = \lvert 2-r\rvert < 1$ |
| Bifurcación | en $r = 3$, calculable a mano |
| Cascada de duplicaciones | el diagrama completo |
| Autosemejanza | los zooms |
| Universalidad | $\delta \approx 4.67$, igual para el mapa seno |

Tres ideas para cerrar:

1. **La complejidad no necesita ecuaciones complicadas.** Todo lo de hoy salió de
   $x(1-x)$, que es lo más simple que hay después de una recta, y empezó con una
   calculadora de bolsillo.

2. **Hay orden adentro del caos, y es medible.** El diagrama de bifurcación no es
   un borrón: tiene ventanas, tiene estructura repetida a todas las escalas, y tiene
   un número asociado.

3. **Ese número es universal.** $\delta = 4.669$ no es una propiedad de la
   parábola sino de la *forma* de llegar al caos, y se mide en experimentos de
   verdad, en sistemas que no tienen nada que ver entre sí.

> **Para el aula:** la parte de la calculadora funciona muy bien repartiendo
> distintos valores de $r$ a distintos grupos y poniendo los resultados en común.
> El diagrama de bifurcación se puede armar en una planilla de cálculo. Y la
> discusión sobre por qué dos personas que hacen la misma cuenta obtienen números
> distintos a partir del paso quince es una clase entera de sensibilidad a las
> condiciones iniciales, sin una sola ecuación diferencial.

**Mañana** volvemos a las ecuaciones diferenciales, ahora en tres dimensiones, con
el sistema de Lorenz. Vamos a medir el caos con un número y, sobre el final, vamos
a encontrar escondido adentro del atractor un mapa de una sola variable,
sospechosamente parecido al de hoy.